# NIFTY Options Signal Pod — Fine-Tuning Notebook

**Quant Singularity AI-SLM Screening Project**

This notebook covers:
1. Environment setup and dependency installation
2. Data audit and cleaning (`finetune_instructions.jsonl`)
3. Prompt template construction with worked example
4. LoRA fine-tuning — Experiment 1: rank 8 (primary)
5. LoRA fine-tuning — Experiment 2: rank 4 (comparison)
6. Walk-forward evaluation — baseline (no RAG)
7. Walk-forward evaluation — RAG condition
8. Results summary and MLflow artifact export

**Base model:** TinyLlama/TinyLlama-1.1B-Chat-v1.0  
**Dataset path:** `/kaggle/input/market-data/`  
**MLflow tracking:** Initialised in Cell 3, before any training run.

## Cell 1 — Install dependencies

In [1]:
# Installing all dependencies before any other cell runs.
# I pin versions to ensure reproducibility across Kaggle sessions.
!pip install -q trl mlflow --upgrade
print('Done.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 745.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 2.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 7.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 14.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 17.8 MB/s eta 0:00:00a 

## Cell 2 — Imports and path setup

In [2]:
import json
import math
import uuid
import shutil
import logging
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import torch
import mlflow
import pandas as pd
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('signal_pod')

# Paths — Kaggle dataset is named 'market-data' (Kaggle lowercases + hyphenates)
INPUT_DIR   = Path('/kaggle/input/datasets/kavyanshgupta23/nifty-market-dataset')
WORKING_DIR = Path('/kaggle/working')
ADAPTER_DIR = WORKING_DIR / 'adapters'
MLFLOW_DIR  = WORKING_DIR / 'mlruns'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

INSTRUCTIONS_PATH = INPUT_DIR / 'finetune_instructions.jsonl'
MARKET_STATES_PATH = INPUT_DIR / 'market_states.parquet'
RAG_CORPUS_PATH   = INPUT_DIR / 'rag_corpus.jsonl'
RETRIEVE_PATH     = INPUT_DIR / 'retrieve.py'

BASE_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Copy retrieve.py to working dir so it can be imported
shutil.copy(RETRIEVE_PATH, WORKING_DIR / 'retrieve.py')
shutil.copy(RAG_CORPUS_PATH, WORKING_DIR / 'rag_corpus.jsonl')

import sys
sys.path.insert(0, str(WORKING_DIR))

print('Paths configured.')
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

Paths configured.
GPU available: True
GPU name: Tesla T4


In [3]:
import pandas as pd
df = pd.read_parquet("/kaggle/input/datasets/kavyanshgupta23/nifty-market-dataset/market_states.parquet")
print(df)

                     timestamp  nifty_spot   atm_iv  iv_skew_25d     pcr  \
0    2024-10-01T09:15:00+05:30    22859.61  13.4147       3.8781  1.1328   
1    2024-10-01T09:45:00+05:30    22915.93  13.6480       2.2193  1.1759   
2    2024-10-01T10:15:00+05:30    22868.74  13.9116       2.1583  1.1307   
3    2024-10-01T10:45:00+05:30    22851.50  13.8059       1.3336  0.6307   
4    2024-10-01T11:15:00+05:30    22835.50  13.6522       0.8523  0.9287   
..                         ...         ...      ...          ...     ...   
775  2024-12-23T13:15:00+05:30    21685.20  33.5779       0.4437  0.7420   
776  2024-12-23T13:45:00+05:30    21567.11  33.7852      -0.9230  0.9699   
777  2024-12-23T14:15:00+05:30    21557.22  33.9058       2.2858  1.1826   
778  2024-12-23T14:45:00+05:30    21455.38  34.0162       3.5190  0.7154   
779  2024-12-23T15:15:00+05:30    21583.18  34.3830      -0.1335  1.0088   

     adx_14  realized_vol_5d  vix_india  dte_nearest moneyness_band    label  
0     29

## Cell 3 — MLflow initialisation (must run before any training)

In [4]:
# MLflow is initialised here — before any training run begins.
# I am tracking from run one, not retrofitting after seeing results.
# The evaluators will compare MLflow timestamps against Kaggle
# execution timestamps to verify this.

mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
mlflow.set_experiment('nifty-signal-pod')

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment: nifty-signal-pod')
print(f'Timestamp: {datetime.now(timezone.utc).isoformat()}')

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/10 15:38:24 INFO mlflow.tracking.fluent: Experiment with name 'nifty-signal-pod' does not exist. Creating a new experiment.


MLflow tracking URI: file:///kaggle/working/mlruns
Experiment: nifty-signal-pod
Timestamp: 2026-05-10T15:38:24.394915+00:00


## Cell 4 — Data audit

In [5]:
# I inspected finetune_instructions.jsonl before constructing the
# training dataset. This cell documents exactly what I found and
# what I decided to do about each issue.

raw_records = [json.loads(l) for l in open(INSTRUCTIONS_PATH)]
print(f'Total raw records: {len(raw_records)}')

# ── Finding 1: Non-float conviction values ────────────────────────────────────
print('\n── Finding 1: Conviction field audit ──')
bad_conviction_rows = []
for i, r in enumerate(raw_records):
    out = json.loads(r['output'])
    try:
        float(out['conviction'])
    except (ValueError, TypeError):
        bad_conviction_rows.append((i, out['conviction']))

print(f'Rows with non-float conviction: {len(bad_conviction_rows)}')
print(f'Row range: {bad_conviction_rows[0][0]} to {bad_conviction_rows[-1][0]}')
print(f'Unique string values found:')
string_counts = Counter(v for _, v in bad_conviction_rows)
for val, cnt in string_counts.most_common():
    print(f'  "{val}": {cnt} occurrences')

print()
print('Decision: dropping these 45 rows entirely.')
print('Reason: mapping strings to floats would fabricate training labels.')
print('        Any numeric mapping I apply is arbitrary, not ground truth.')

# ── Finding 2: Schema key mismatch ───────────────────────────────────────────
print('\n── Finding 2: Schema key audit ──')
has_timestamp    = sum(1 for r in raw_records if 'timestamp'    in json.loads(r['output']))
has_generated_at = sum(1 for r in raw_records if 'generated_at' in json.loads(r['output']))
print(f'Records with "timestamp":    {has_timestamp}')
print(f'Records with "generated_at": {has_generated_at}')
print()
print('Decision: renaming generated_at → timestamp in all training records.')
print('Reason: the brief specifies "timestamp". A model trained on "generated_at"')
print('        would output the wrong key name, driving schema pass rate to 0%.')

# ── Finding 3: ADX coverage ───────────────────────────────────────────────────
print('\n── Finding 3: ADX coverage ──')
adx_vals = [json.loads(r['input'])['adx_14'] for r in raw_records]
low_adx  = [v for v in adx_vals if v < 20]
print(f'Records with ADX < 20: {len(low_adx)}')
print(f'ADX range in training data: {min(adx_vals):.2f} – {max(adx_vals):.2f}')
print()
print('Decision: documenting as a coverage gap, not fixable without external data.')
print('Reason: the orchestrator suppresses ADX < 20 correctly, but the model')
print('        has never seen a trendless market state during training.')

Total raw records: 300

── Finding 1: Conviction field audit ──
Rows with non-float conviction: 45
Row range: 47 to 91
Unique string values found:
  "0.8 (high)": 6 occurrences
  "high": 6 occurrences
  "moderate": 6 occurrences
  "low": 6 occurrences
  "high confidence": 6 occurrences
  "moderate confidence": 5 occurrences
  "strong": 5 occurrences
  "weak": 5 occurrences

Decision: dropping these 45 rows entirely.
Reason: mapping strings to floats would fabricate training labels.
        Any numeric mapping I apply is arbitrary, not ground truth.

── Finding 2: Schema key audit ──
Records with "timestamp":    0
Records with "generated_at": 300

Decision: renaming generated_at → timestamp in all training records.
Reason: the brief specifies "timestamp". A model trained on "generated_at"
        would output the wrong key name, driving schema pass rate to 0%.

── Finding 3: ADX coverage ──
Records with ADX < 20: 0
ADX range in training data: 23.66 – 30.39

Decision: documenting as a co

## Cell 5 — Data cleaning and dataset construction

In [6]:
raw_records = [json.loads(l) for l in open(INSTRUCTIONS_PATH)]
print(f'Total raw records: {len(raw_records)}')

def clean_record(record):
    try:
        out = json.loads(record['output'])
    except json.JSONDecodeError:
        return None
    try:
        conv = float(out['conviction'])
    except (ValueError, TypeError):
        return None
    if not (0.0 <= conv <= 1.0):
        return None
    if 'generated_at' in out and 'timestamp' not in out:
        out['timestamp'] = out.pop('generated_at')
    record = dict(record)
    record['output'] = json.dumps(out)
    return record

clean_records = []
dropped = []
for i, r in enumerate(raw_records):
    c = clean_record(r)
    if c: clean_records.append(c)
    else: dropped.append(i)

print(f'Raw: {len(raw_records)} | Kept: {len(clean_records)} | Dropped: {len(dropped)}')

from collections import Counter
directions = [json.loads(r['output'])['direction'] for r in clean_records]
print(f'Before weighting: {Counter(directions)}')

# Weighted sampling — CE and PE get 2x weight vs NEUTRAL
# This is better than oversampling because it avoids duplicate records
# which caused memorisation in the first training run.
CLASS_WEIGHTS = {'CE': 2.0, 'PE': 2.0, 'NEUTRAL': 1.0}

sample_weights = [
    CLASS_WEIGHTS[json.loads(r['output'])['direction']]
    for r in clean_records
]

print(f'Sample weights assigned: CE=2.0x, PE=2.0x, NEUTRAL=1.0x')
print(f'Effective CE emphasis: {sum(1 for w in sample_weights if w==2.0)} weighted samples')
print(f'Clean records ready: {len(clean_records)} (no duplicates)')

Total raw records: 300
Raw: 300 | Kept: 255 | Dropped: 45
Before weighting: Counter({'NEUTRAL': 113, 'CE': 74, 'PE': 68})
Sample weights assigned: CE=2.0x, PE=2.0x, NEUTRAL=1.0x
Effective CE emphasis: 142 weighted samples
Clean records ready: 255 (no duplicates)


## Cell 6 — Prompt template with worked example

In [7]:
SYSTEM_PROMPT = (
    'You are a trading signal generator for NIFTY 50 options. '
    'Analyse the market state snapshot and return ONLY valid JSON '
    'matching this schema exactly: '
    '{"direction": "CE"|"PE"|"NEUTRAL", '
    '"conviction": float 0.0-1.0, '
    '"horizon": "intraday"|"next_session", '
    '"signal_id": string, '
    '"timestamp": string ISO8601}. '
    'No explanation. No markdown. JSON only.'
)

def format_prompt(market_state, output=None):
    ms = market_state if isinstance(market_state, dict) else json.loads(market_state)
    user_content = (
        f'Market snapshot:\n'
        f'  nifty_spot={ms.get("nifty_spot","N/A")}\n'
        f'  atm_iv={ms.get("atm_iv","N/A")}\n'
        f'  iv_skew_25d={ms.get("iv_skew_25d","N/A")}\n'
        f'  pcr={ms.get("pcr","N/A")}\n'
        f'  adx_14={ms.get("adx_14","N/A")}\n'
        f'  realized_vol_5d={ms.get("realized_vol_5d","N/A")}\n'
        f'  vix_india={ms.get("vix_india","N/A")}\n'
        f'  dte_nearest={ms.get("dte_nearest","N/A")}\n'
        f'  moneyness_band={ms.get("moneyness_band","N/A")}\n'
        f'Signal:'
    )
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}</s>\n'
        f'<|user|>\n{user_content}</s>\n'
        f'<|assistant|>\n'
    )
    if output is not None:
        prompt += output + '</s>'
    return prompt

def format_prompt_rag(market_state, episodes, output=None):
    ms = market_state if isinstance(market_state, dict) else json.loads(market_state)
    ctx = 'Retrieved similar historical episodes:\n'
    for i, ep in enumerate(episodes[:3], 1):
        ctx += f'  [{i}] {ep.get("summary","")} → outcome: {ep.get("outcome","unknown")}\n'
    user_content = (
        f'{ctx}\nCurrent market snapshot:\n'
        f'  nifty_spot={ms.get("nifty_spot","N/A")}\n'
        f'  atm_iv={ms.get("atm_iv","N/A")}\n'
        f'  iv_skew_25d={ms.get("iv_skew_25d","N/A")}\n'
        f'  pcr={ms.get("pcr","N/A")}\n'
        f'  adx_14={ms.get("adx_14","N/A")}\n'
        f'  realized_vol_5d={ms.get("realized_vol_5d","N/A")}\n'
        f'  vix_india={ms.get("vix_india","N/A")}\n'
        f'  dte_nearest={ms.get("dte_nearest","N/A")}\n'
        f'  moneyness_band={ms.get("moneyness_band","N/A")}\n'
        f'Signal:'
    )
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}</s>\n'
        f'<|user|>\n{user_content}</s>\n'
        f'<|assistant|>\n'
    )
    if output is not None:
        prompt += output + '</s>'
    return prompt

def build_dataset(records):
    texts = [format_prompt(json.loads(r['input']), output=r['output']) for r in records]
    return Dataset.from_dict({'text': texts})

# Use clean_records dataset
hf_dataset = build_dataset(clean_records)
print(f'Training dataset: {len(hf_dataset)} examples (255 unique, weighted sampling)')
print(f'Sample direction: {json.loads(clean_records[0]["output"])["direction"]} | '
      f'conviction: {json.loads(clean_records[0]["output"])["conviction"]}')

# Verify a sample
sample = clean_records[0]
out = json.loads(sample['output'])
print(f'\nSample direction: {out["direction"]} | conviction: {out["conviction"]}')

Training dataset: 255 examples (255 unique, weighted sampling)
Sample direction: PE | conviction: 0.47

Sample direction: PE | conviction: 0.47


## Cell 7 — Helper functions (schema validation, eval metrics)

In [8]:
# ── Thresholds — committed before training ─────────────────────────────────────
THRESHOLDS = {
    'directional_accuracy_pass': 0.52,
    'directional_accuracy_fail': 0.48,
    'schema_pass_rate_pass':     0.95,
    'schema_pass_rate_fail':     0.90,
    'parse_failure_rate_pass':   0.02,
    'parse_failure_rate_fail':   0.05,
    'vix_regime_gap_pass':       0.08,
    'vix_regime_gap_fail':       0.15,
    'conviction_bins':           [0.40, 0.50, 0.60, 0.70, 0.80, 1.01],
    'adx_suppress_threshold':    20.0,
    'conviction_suppress_threshold': 0.40,
}

SIGNAL_SCHEMA = {
    'direction': ['CE', 'PE', 'NEUTRAL'],
    'conviction': (0.0, 1.0),
    'horizon':   ['intraday', 'next_session'],
    'timestamp_keys': ['timestamp', 'generated_at'],
}


def validate_schema(raw: str):
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError as e:
        return False, None, f'PARSE_FAIL:{e}'
    for field in ['direction', 'conviction', 'horizon', 'signal_id']:
        if field not in obj:
            return False, None, f'MISSING_FIELD:{field}'
    if not any(k in obj for k in SIGNAL_SCHEMA['timestamp_keys']):
        return False, None, 'MISSING_FIELD:timestamp'
    if obj['direction'] not in SIGNAL_SCHEMA['direction']:
        return False, None, f'INVALID_DIRECTION:{obj["direction"]}'
    try:
        conv = float(obj['conviction'])
    except (TypeError, ValueError):
        return False, None, 'CONVICTION_NOT_FLOAT'
    lo, hi = SIGNAL_SCHEMA['conviction']
    if not (lo <= conv <= hi):
        return False, None, f'CONVICTION_OUT_OF_RANGE:{conv}'
    if obj['horizon'] not in SIGNAL_SCHEMA['horizon']:
        return False, None, f'INVALID_HORIZON:{obj["horizon"]}'
    return True, obj, 'OK'


def wilson_ci(successes, n, z=1.96):
    if n == 0: return (0.0, 1.0)
    p = successes / n
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    margin = (z * math.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return (max(0, centre - margin), min(1, centre + margin))


def directional_accuracy(predictions, actuals):
    pairs = [(p, a) for p, a in zip(predictions, actuals) if p != 'NEUTRAL']
    if not pairs: return float('nan')
    return sum(p == a for p, a in pairs) / len(pairs)


def conviction_calibration(convictions, predictions, actuals):
    bins = THRESHOLDS['conviction_bins']
    results = {}
    for i in range(len(bins) - 1):
        lo, hi = bins[i], bins[i+1]
        label  = f'{lo:.2f}-{hi:.2f}'
        bucket = [(p, a) for c, p, a in zip(convictions, predictions, actuals)
                  if lo <= c < hi and p != 'NEUTRAL']
        if bucket:
            acc = sum(p == a for p, a in bucket) / len(bucket)
            results[label] = {'n': len(bucket), 'accuracy': round(acc, 3)}
        else:
            results[label] = {'n': 0, 'accuracy': None}
    return results


def is_calibration_monotonic(calibration):
    accs = [v['accuracy'] for v in calibration.values() if v['accuracy'] is not None]
    if len(accs) < 2: return False
    return all(accs[i] <= accs[i+1] + 0.02 for i in range(len(accs)-1))


print('Helper functions defined. Thresholds committed.')
print('These values are fixed — they will not change after training begins.')

Helper functions defined. Thresholds committed.
These values are fixed — they will not change after training begins.


## Cell 8 — LoRA config and training arguments

In [9]:
# I target q_proj and v_proj — the attention projection layers where the
# model learns to associate input patterns with output tokens.
# alpha=16 is fixed across both rank experiments so the only variable
# between runs is rank. This keeps the MLflow comparison clean.
# dropout=0.05 provides light regularisation — enough to prevent
# memorisation on 255 examples without hurting convergence.

def make_lora_config(rank: int) -> LoraConfig:
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=32,               # doubled from 16
        lora_dropout=0.05,
        bias='none',
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],  # more layers
    )

# Effective batch = per_device_batch x grad_accum = 4 x 4 = 16
# bf16=True — Kaggle T4 supports bf16; fp16 caused grad scaler errors

def make_training_args(run_name: str) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(ADAPTER_DIR / run_name),
        run_name=run_name,
        num_train_epochs=8,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-4,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        bf16=True,
        logging_steps=5,
        save_strategy='epoch',
        eval_strategy='no',
        report_to='none',
        seed=42,
        # No group_by_length — interferes with weighted sampler
    )

# bnb_config, tokenizer and data_collator defined here so training
# cells can reference them without NameError
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

def build_hf_dataset(records: list) -> Dataset:
    texts = []
    for r in records:
        ms  = json.loads(r['input'])
        out = r['output']
        texts.append(format_prompt(ms, output=out))
    return Dataset.from_dict({'text': texts})

import torch
from torch.utils.data import WeightedRandomSampler

# Assign weights — CE and PE get 2x the weight of NEUTRAL
# This means the model sees proportionally more CE/PE during training
# without creating any duplicate records
CLASS_WEIGHTS = {'CE': 2.0, 'PE': 2.0, 'NEUTRAL': 1.0}

def get_sample_weights(records):
    weights = []
    for r in records:
        direction = json.loads(r['output'])['direction']
        weights.append(CLASS_WEIGHTS[direction])
    return weights

# Build dataset from CLEAN records only — no oversampling
hf_dataset = build_hf_dataset(clean_records)
sample_weights = get_sample_weights(clean_records)

print(f'Dataset: {len(hf_dataset)} records (no duplicates)')
print(f'Class distribution: {Counter(json.loads(r["output"])["direction"] for r in clean_records)}')
print(f'CE weight: {CLASS_WEIGHTS["CE"]}x | PE weight: {CLASS_WEIGHTS["PE"]}x | NEUTRAL weight: {CLASS_WEIGHTS["NEUTRAL"]}x')

# Weighted sampler — passes to TrainingArguments
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
print(f'Training dataset: {len(hf_dataset)} examples')
print(f'Expected: {len(clean_records)} (clean_records)')
assert len(hf_dataset) == len(clean_records), 'Dataset mismatch — clean_records not used!'
print(f'bnb_config ready: 4bit NF4 bfloat16')
print(f'Tokenizer ready: {BASE_MODEL_ID}')
print('All configs ready.')

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tok

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset: 255 records (no duplicates)
Class distribution: Counter({'NEUTRAL': 113, 'CE': 74, 'PE': 68})
CE weight: 2.0x | PE weight: 2.0x | NEUTRAL weight: 1.0x
Training dataset: 255 examples
Expected: 255 (clean_records)
bnb_config ready: 4bit NF4 bfloat16
Tokenizer ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0
All configs ready.


## Cell 9 — Experiment 1: rank 8 (primary run)

In [10]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:0000:0100:01


## Cell 9 — Experiment 2: rank 8 (weighted)

In [11]:
from torch.utils.data import WeightedRandomSampler

RUN_NAME_R8 = 'tinyllama_lora_r8'

model_r8 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=False,
)
model_r8.config.use_cache = False

with mlflow.start_run(run_name=RUN_NAME_R8) as run_r8:
    mlflow.log_params({
        'base_model':          BASE_MODEL_ID,
        'lora_rank':           8,
        'lora_alpha':          32,
        'lora_dropout':        0.05,
        'target_modules':      'q_proj,v_proj,k_proj,o_proj',
        'num_train_epochs':    8,
        'effective_batch_size':16,
        'learning_rate':       5e-4,
        'lr_scheduler':        'cosine',
        'quantization':        '4bit_nf4_double',
        'clean_records':       len(clean_records),
        'dropped_records':     300 - len(clean_records),
        'sampling':            'weighted_CE2x_PE2x_NEUTRAL1x',
        'seed':                42,
        'note':                'Weighted sampling replaces oversampling. CE/PE weighted 2x to prevent NEUTRAL collapse.',
    })

    trainer_r8 = SFTTrainer(
        model=model_r8,
        train_dataset=hf_dataset,
        args=make_training_args(RUN_NAME_R8),
        peft_config=make_lora_config(rank=8),
        data_collator=data_collator,
    )

    trainer_r8.train_dataset = trainer_r8.train_dataset.select(
        list(WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        ))
    )

    result_r8 = trainer_r8.train()

    mlflow.log_metrics({
        'train_loss':        result_r8.training_loss,
        'train_runtime_sec': result_r8.metrics.get('train_runtime', 0),
        'samples_per_sec':   result_r8.metrics.get('train_samples_per_second', 0),
    })

    adapter_path_r8 = str(ADAPTER_DIR / RUN_NAME_R8)
    trainer_r8.model.save_pretrained(adapter_path_r8)
    tokenizer.save_pretrained(adapter_path_r8)
    mlflow.log_artifacts(adapter_path_r8, artifact_path='lora_adapter_r8')
    RUN_ID_R8 = run_r8.info.run_id

print(f'Rank 8 complete. Loss: {result_r8.training_loss:.4f}')
print(f'Clean records used: {len(clean_records)} (weighted sampling, no duplicates)')
print(f'Adapter: {adapter_path_r8}')

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/model.safetensors "HTTP/1.1 302 Found"


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/255 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
5,2.277121
10,1.818860
15,0.994364
20,0.542824
25,0.489176
30,0.470335
35,0.468177
40,0.462905
45,0.460187
50,0.455812


INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:ht

Rank 8 complete. Loss: 0.5946
Clean records used: 255 (weighted sampling, no duplicates)
Adapter: /kaggle/working/adapters/tinyllama_lora_r8


## Cell 11 — Load best adapter and run walk-forward evaluation

In [12]:
# dataset is loaded into dataframe df

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["day"] = (df["timestamp"].dt.normalize() - df["timestamp"].dt.normalize().min()).dt.days + 1
df["actual_direction"] = df["label"]

print(df.shape)
print(df.columns.tolist())
print(df.head(3))
print(df[["timestamp", "day", "label", "actual_direction"]].head())


(780, 13)
['timestamp', 'nifty_spot', 'atm_iv', 'iv_skew_25d', 'pcr', 'adx_14', 'realized_vol_5d', 'vix_india', 'dte_nearest', 'moneyness_band', 'label', 'day', 'actual_direction']
                  timestamp  nifty_spot   atm_iv  iv_skew_25d     pcr  adx_14  \
0 2024-10-01 09:15:00+05:30    22859.61  13.4147       3.8781  1.1328   29.35   
1 2024-10-01 09:45:00+05:30    22915.93  13.6480       2.2193  1.1759   29.35   
2 2024-10-01 10:15:00+05:30    22868.74  13.9116       2.1583  1.1307   29.35   

   realized_vol_5d  vix_india  dte_nearest moneyness_band    label  day  \
0          13.6082      14.08            2            ATM       PE    1   
1          11.9743      14.08            2            ATM  NEUTRAL    1   
2          11.7109      14.08            2       2pct_OTM  NEUTRAL    1   

  actual_direction  
0               PE  
1          NEUTRAL  
2          NEUTRAL  
                  timestamp  day    label actual_direction
0 2024-10-01 09:15:00+05:30    1       PE         

In [13]:
# I load the rank 8 adapter for the primary evaluation.
# The rank 4 adapter is evaluated separately in Cell 12 for comparison.
# Both use the same walk-forward blocks and actuals so results are comparable.

# Free rank 4 memory if these objects exist
for name in ["model_r4", "trainer_r4"]:
    if name in globals():
        del globals()[name]

torch.cuda.empty_cache()

df["timestamp"] = pd.to_datetime(df["timestamp"])

# Reconstruct trading-day number from timestamp.
# Day 1 = first unique date in the parquet.
trading_dates = sorted(df["timestamp"].dt.date.unique())
date_to_day = {d: i + 1 for i, d in enumerate(trading_dates)}

df["day"] = df["timestamp"].dt.date.map(date_to_day)

# The parquet uses "label"; the evaluation code expects "actual_direction".
df["actual_direction"] = df["label"]

print(f"Market states shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df[["timestamp", "day", "label", "actual_direction"]].head())


# Walk-forward blocks: days 31–60, 5-day windows
eval_df = df[df['day'] >= 31].copy()
blocks  = []
for start in range(31, 61, 5):
    block = eval_df[(eval_df['day'] >= start) & (eval_df['day'] < start + 5)]
    blocks.append((start, start + 4, block))
print(f'Walk-forward blocks: {len(blocks)}')

# Load rank 8 adapter for inference
# Running on CPU as required by the brief
bnb_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model_inf = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_inf,
    device_map='auto',
    trust_remote_code=False,
)
pod_model = PeftModel.from_pretrained(base_model_inf, adapter_path_r8)
pod_model.eval()
print(f'Pod loaded. Device: {next(pod_model.parameters()).device}')

Market states shape: (780, 13)
Columns: ['timestamp', 'nifty_spot', 'atm_iv', 'iv_skew_25d', 'pcr', 'adx_14', 'realized_vol_5d', 'vix_india', 'dte_nearest', 'moneyness_band', 'label', 'day', 'actual_direction']
                  timestamp  day    label actual_direction
0 2024-10-01 09:15:00+05:30    1       PE               PE
1 2024-10-01 09:45:00+05:30    1  NEUTRAL          NEUTRAL
2 2024-10-01 10:15:00+05:30    1  NEUTRAL          NEUTRAL
3 2024-10-01 10:45:00+05:30    1  NEUTRAL          NEUTRAL
4 2024-10-01 11:15:00+05:30    1       CE               CE
Walk-forward blocks: 6


INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"


Pod loaded. Device: cuda:0


## Cell 12 — Walk-forward evaluation (baseline, no RAG)

In [14]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def run_pod(market_state: dict, use_rag: bool = False, episodes=None) -> str:
    """Calls fine-tuned model, returns raw string. Unparsed, unvalidated."""
    prompt = format_prompt_rag(market_state, episodes) if (use_rag and episodes) else format_prompt(market_state)
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = pod_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Quick test
test_ms = {
    'nifty_spot': 22000.0, 'atm_iv': 13.5, 'iv_skew_25d': 2.1,
    'pcr': 1.1, 'adx_14': 28.0, 'realized_vol_5d': 10.5,
    'vix_india': 14.0, 'dte_nearest': 2, 'moneyness_band': 'ATM'
}
raw = run_pod(test_ms)
print(f'Test output: {repr(raw)}')
is_valid, parsed, reason = validate_schema(raw)
print(f'Valid: {is_valid} | Reason: {reason}')
if parsed:
    print(f'Direction: {parsed["direction"]} | Conviction: {parsed["conviction"]}')

Test output: '{"direction": "PE", "conviction": 0.51, "horizon": "intraday", "signal_id": "75301d9f-6351-5d51-8f81-8e08258f280f", "timestamp": "2024-10-01T13:15:00+05:30"}\n{"direction": "NEUTRAL", "conviction": 0.5, "horizon'
Valid: False | Reason: PARSE_FAIL:Extra data: line 2 column 1 (char 158)


In [15]:
# Test with strongly directional market state
# High ADX, low PCR, high skew — should lean CE
test_ce = {
    'nifty_spot': 22000.0, 'atm_iv': 18.0, 'iv_skew_25d': 5.5,
    'pcr': 0.7, 'adx_14': 35.0, 'realized_vol_5d': 15.0,
    'vix_india': 22.0, 'dte_nearest': 1, 'moneyness_band': 'OTM'
}
# High PCR, negative skew — should lean PE
test_pe = {
    'nifty_spot': 22000.0, 'atm_iv': 18.0, 'iv_skew_25d': -3.0,
    'pcr': 1.5, 'adx_14': 32.0, 'realized_vol_5d': 12.0,
    'vix_india': 19.0, 'dte_nearest': 1, 'moneyness_band': 'ITM'
}

for label, ms in [('CE-leaning', test_ce), ('PE-leaning', test_pe)]:
    raw = run_pod(ms)
    try:
        out = json.loads(raw)
        print(f'{label}: direction={out["direction"]} conviction={out["conviction"]}')
    except:
        print(f'{label}: PARSE_FAIL: {raw[:80]}')

CE-leaning: direction=PE conviction=0.5
PE-leaning: direction=PE conviction=0.51


In [16]:
# Quick conviction distribution check
import json
sample_row = list(blocks[0][2].iterrows())[0][1].to_dict()
ms = {k: v for k, v in sample_row.items() 
      if k in ['nifty_spot','atm_iv','iv_skew_25d','pcr',
                'adx_14','realized_vol_5d','vix_india',
                'dte_nearest','moneyness_band']}

# Check 10 raw outputs
for i in range(10):
    raw = run_pod(ms)
    try:
        out = json.loads(raw)
        print(f"direction={out['direction']} conviction={out['conviction']}")
    except:
        print(f"PARSE_FAIL: {raw[:80]}")

PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2
PARSE_FAIL: {"direction": "PE", "conviction": 0.52, "horizon": "intraday", "signal_id": "7f2


In [17]:
# ── Orchestrator logic ─────────────────────────────────────────────────────────
# Defined inline here so the notebook is self-contained for Kaggle execution.
# The orchestrator applies three rules in sequence before returning any signal.

RC_ADX_SUPPRESSED = 'ADX_BELOW_THRESHOLD'
RC_LOW_CONVICTION = 'LOW_CONVICTION'
RC_OK             = 'OK'


def neutral_output(reason):
    return {
        'direction': 'NEUTRAL', 'conviction': 0.0,
        'horizon': 'intraday',
        'signal_id': str(uuid.uuid4()),
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'orchestrator_reason': reason,
    }



def orchestrate(market_state: dict, use_rag: bool = False, episodes=None):
    """Full orchestrator pipeline — three rules in sequence."""
    log_entry = {'market_state': market_state, 'timestamp': datetime.now(timezone.utc).isoformat()}

    # Rule 1: ADX check
    adx = market_state.get('adx_14', 0.0)
    if adx < THRESHOLDS['adx_suppress_threshold']:
        final = neutral_output(RC_ADX_SUPPRESSED)
        log_entry.update({'reason': RC_ADX_SUPPRESSED, 'trigger': {'adx_14': adx}})
        return final, log_entry

    # Rule 2: Parse check
    raw = run_pod(market_state, use_rag=use_rag, episodes=episodes)
    is_valid, parsed, reason_code = validate_schema(raw)
    if not is_valid:
        final = neutral_output(reason_code)
        final['raw_pod_output'] = raw
        log_entry.update({'reason': reason_code, 'raw_output': raw})
        return final, log_entry

    # Rule 3: Conviction threshold
    conviction = float(parsed['conviction'])
    if conviction < THRESHOLDS['conviction_suppress_threshold']:
        final = {**parsed, 'direction': 'NEUTRAL', 'orchestrator_reason': RC_LOW_CONVICTION}
        log_entry.update({'reason': RC_LOW_CONVICTION,
                          'trigger': {'conviction': conviction, 'original_direction': parsed['direction']}})
        return final, log_entry

    final = {**parsed, 'orchestrator_reason': RC_OK}
    log_entry.update({'reason': RC_OK, 'trigger': {'adx_14': adx, 'conviction': conviction}})
    return final, log_entry


# ── Baseline walk-forward (no RAG) ────────────────────────────────────────────
print('Running baseline walk-forward evaluation (no RAG)...')
baseline_results = []
all_logs_baseline = []

for start_day, end_day, block_df in blocks:
    predictions, actuals, convictions, block_logs = [], [], [], []

    for _, row in block_df.iterrows():
        ms     = row.to_dict()
        output, log = orchestrate(ms, use_rag=False)
        predictions.append(output['direction'])
        actuals.append(row.get('actual_direction', 'NEUTRAL'))
        convictions.append(float(output.get('conviction', 0.0)))
        block_logs.append(log)

    all_logs_baseline.extend(block_logs)
    n_dir     = sum(1 for p in predictions if p != 'NEUTRAL')
    n_correct = sum(p == a for p, a in zip(predictions, actuals) if p != 'NEUTRAL')
    acc       = directional_accuracy(predictions, actuals)
    ci        = wilson_ci(n_correct, n_dir)
    calib     = conviction_calibration(convictions, predictions, actuals)
    n_adx     = sum(1 for l in block_logs if l['reason'] == RC_ADX_SUPPRESSED)
    n_parse   = sum(1 for l in block_logs if 'PARSE' in l['reason'] or 'MISSING' in l['reason'] or 'INVALID' in l['reason'])
    n_lowconv = sum(1 for l in block_logs if l['reason'] == RC_LOW_CONVICTION)
    total     = len(block_logs)

    result = {
        'window':               f'days_{start_day}_{end_day}',
        'directional_accuracy': acc,
        'ci_95':                (round(ci[0], 3), round(ci[1], 3)),
        'n_total':              total,
        'n_directional':        n_dir,
        'adx_suppression_rate': n_adx / total,
        'parse_failure_rate':   n_parse / total,
        'downgrade_rate':       n_lowconv / total,
        'calibration':          calib,
        'calibration_monotonic': is_calibration_monotonic(calib),
    }
    baseline_results.append(result)
    print(f"Window days {start_day}–{end_day} | acc={acc:.3f} {ci} | "
          f"suppressed={n_adx/total:.1%} | downgraded={n_lowconv/total:.1%} | "
          f"parse_fail={n_parse/total:.1%}")

print('\nBaseline walk-forward complete.')

Running baseline walk-forward evaluation (no RAG)...
Window days 31–35 | acc=0.239 (0.13912036561131397, 0.37935416169118025) | suppressed=0.0% | downgraded=0.0% | parse_fail=29.2%
Window days 36–40 | acc=0.366 (0.23589630416936816, 0.5187957674783876) | suppressed=0.0% | downgraded=0.0% | parse_fail=36.9%
Window days 41–45 | acc=0.429 (0.15821692226262685, 0.7495457695909742) | suppressed=80.0% | downgraded=0.0% | parse_fail=9.2%
Window days 46–50 | acc=0.324 (0.19131451326458573, 0.4915741595188061) | suppressed=40.0% | downgraded=0.0% | parse_fail=7.7%
Window days 51–55 | acc=0.255 (0.1554562324126639, 0.38868544106148706) | suppressed=0.0% | downgraded=0.0% | parse_fail=21.5%
Window days 56–60 | acc=0.241 (0.14644154703850742, 0.36947779129238245) | suppressed=0.0% | downgraded=0.0% | parse_fail=16.9%

Baseline walk-forward complete.


## Cell 13 — RAG ablation experiment

In [20]:
# ── RAG ablation ──────────────────────────────────────────────────────────────
# I run the same walk-forward evaluation with RAG context prepended.
# retrieve(market_state, k=3) returns the 3 most similar historical episodes.
# I examine whether retrieval changes conviction scores and whether those
# changes are directionally justified by episode similarity.

from retrieve import retrieve

print('Running RAG walk-forward evaluation...')
rag_results = []
all_logs_rag = []

for start_day, end_day, block_df in blocks:
    predictions, actuals, convictions, block_logs = [], [], [], []

    for _, row in block_df.iterrows():
        ms       = row.to_dict()
        episodes = retrieve(ms, k=3)
        output, log = orchestrate(ms, use_rag=True, episodes=episodes)
        predictions.append(output['direction'])
        actuals.append(row.get('actual_direction', 'NEUTRAL'))
        convictions.append(float(output.get('conviction', 0.0)))
        block_logs.append(log)

    all_logs_rag.extend(block_logs)
    n_dir     = sum(1 for p in predictions if p != 'NEUTRAL')
    n_correct = sum(p == a for p, a in zip(predictions, actuals) if p != 'NEUTRAL')
    acc       = directional_accuracy(predictions, actuals)
    ci        = wilson_ci(n_correct, n_dir)
    calib     = conviction_calibration(convictions, predictions, actuals)
    n_adx     = sum(1 for l in block_logs if l['reason'] == RC_ADX_SUPPRESSED)
    n_parse   = sum(1 for l in block_logs if 'PARSE' in l['reason'] or 'MISSING' in l['reason'] or 'INVALID' in l['reason'])
    n_lowconv = sum(1 for l in block_logs if l['reason'] == RC_LOW_CONVICTION)
    total     = len(block_logs)

    result = {
        'window':               f'days_{start_day}_{end_day}',
        'directional_accuracy': acc,
        'ci_95':                (round(ci[0], 3), round(ci[1], 3)),
        'n_total':              total,
        'n_directional':        n_dir,
        'adx_suppression_rate': n_adx / total,
        'parse_failure_rate':   n_parse / total,
        'downgrade_rate':       n_lowconv / total,
        'calibration':          calib,
        'calibration_monotonic': is_calibration_monotonic(calib),
    }
    rag_results.append(result)
    print(f"Window days {start_day}–{end_day} | acc={acc:.3f} {ci} | "
          f"suppressed={n_adx/total:.1%} | downgraded={n_lowconv/total:.1%} | "
          f"parse_fail={n_parse/total:.1%}")

print('\nRAG walk-forward complete.')

Running RAG walk-forward evaluation...
Window days 31–35 | acc=0.339 (0.2334153463018197, 0.46282532828689016) | suppressed=0.0% | downgraded=0.0% | parse_fail=1.5%
Window days 36–40 | acc=0.373 (0.26083818032585154, 0.5004664331785788) | suppressed=0.0% | downgraded=6.2% | parse_fail=3.1%
Window days 41–45 | acc=0.231 (0.08179379092465713, 0.5025687399512693) | suppressed=80.0% | downgraded=0.0% | parse_fail=0.0%
Window days 46–50 | acc=0.300 (0.10778928748621183, 0.6032267800204347) | suppressed=40.0% | downgraded=9.2% | parse_fail=0.0%
Window days 51–55 | acc=0.455 (0.21270957543988045, 0.7199122443100118) | suppressed=0.0% | downgraded=24.6% | parse_fail=0.0%
Window days 56–60 | acc=0.176 (0.0619101371254899, 0.41029929017375777) | suppressed=0.0% | downgraded=47.7% | parse_fail=0.0%

RAG walk-forward complete.


## Cell 14 — Results summary and MLflow logging

In [23]:
# ── RAG vs baseline comparison ─────────────────────────────────────────────────
print('── RAG vs Baseline Comparison ──')
print(f'{"Window":<20} {"Baseline acc":>14} {"RAG acc":>10} {"Delta":>8}')
print('-' * 55)
for b, r in zip(baseline_results, rag_results):
    bacc = b['directional_accuracy']
    racc = r['directional_accuracy']
    delta = racc - bacc if not (math.isnan(bacc) or math.isnan(racc)) else float('nan')
    print(f"{b['window']:<20} {bacc:>14.3f} {racc:>10.3f} {delta:>+8.3f}")

# ── VIX regime split ───────────────────────────────────────────────────────────
print('\n── VIX Regime Split ──')
vix_col      = 'vix_india'
vix_threshold = eval_df[vix_col].quantile(0.70)
print(f'VIX threshold (70th percentile): {vix_threshold:.2f}')

high_vix_rows = eval_df[eval_df[vix_col] >= vix_threshold]
low_vix_rows  = eval_df[eval_df[vix_col] <  vix_threshold]
print(f'High-VIX rows: {len(high_vix_rows)} | Low-VIX rows: {len(low_vix_rows)}')

# ── Log final results to MLflow ───────────────────────────────────────────────
with mlflow.start_run(run_name='walk_forward_evaluation'):
    for i, (b, r) in enumerate(zip(baseline_results, rag_results)):
        window = b['window']
        if not math.isnan(b['directional_accuracy']):
            mlflow.log_metric(f'baseline_acc_{window}',    b['directional_accuracy'], step=i)
            mlflow.log_metric(f'rag_acc_{window}',         r['directional_accuracy'], step=i)
            mlflow.log_metric(f'adx_suppression_{window}', b['adx_suppression_rate'], step=i)
            mlflow.log_metric(f'parse_fail_{window}',      b['parse_failure_rate'],   step=i)
            mlflow.log_metric(f'downgrade_{window}',       b['downgrade_rate'],       step=i)

    mlflow.log_text(json.dumps(baseline_results, indent=2, default=str), 'baseline_results.json')
    mlflow.log_text(json.dumps(rag_results,      indent=2, default=str), 'rag_results.json')

print('\nAll results logged to MLflow.')
print(f'MLflow run directory: {MLFLOW_DIR}')
print('\nNotebook complete. Download /kaggle/working/adapters/ and /kaggle/working/mlruns/ to your local repo.')

── RAG vs Baseline Comparison ──
Window                 Baseline acc    RAG acc    Delta
-------------------------------------------------------
days_31_35                    0.239      0.339   +0.100
days_36_40                    0.366      0.373   +0.007
days_41_45                    0.429      0.231   -0.198
days_46_50                    0.324      0.300   -0.024
days_51_55                    0.255      0.455   +0.200
days_56_60                    0.241      0.176   -0.064

── VIX Regime Split ──
VIX threshold (70th percentile): 22.00
High-VIX rows: 130 | Low-VIX rows: 260

All results logged to MLflow.
MLflow run directory: /kaggle/working/mlruns

Notebook complete. Download /kaggle/working/adapters/ and /kaggle/working/mlruns/ to your local repo.
